In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    classification_report
)
import joblib

In [2]:
# open train and test data
train_df = pd.read_csv("wildfire_train_split.csv")
test_df  = pd.read_csv("wildfire_test_split.csv")

display(train_df.head())
display(test_df.head())

,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,vs,bi,fm100,fm1000,erc,etr,pet,vpd,Wildfire_binary,cluster_id
0,48.128431,-97.276685,2018-08-15,No,-0.053101,-0.050631,-0.063031,-0.052081,-0.021598,-0.051093,...,-0.052530,-0.048324,-0.053601,-0.053433,-0.045941,-0.050984,-0.051275,-0.051811,0,3
1,48.128431,-97.276685,2018-08-16,No,-0.053101,-0.049571,-0.063620,-0.052080,-0.026678,-0.049967,...,-0.052530,-0.048324,-0.053895,-0.053551,-0.044765,-0.050573,-0.051040,-0.051612,0,3
2,48.128431,-97.276685,2018-08-17,No,-0.053101,-0.055165,-0.059797,-0.052080,-0.025733,-0.048841,...,-0.052471,-0.048324,-0.054189,-0.053609,-0.044765,-0.051161,-0.051393,-0.051858,0,3
3,48.128431,-97.276685,2018-08-18,No,-0.050163,-0.058522,-0.060385,-0.052080,-0.028214,-0.046945,...,-0.051296,-0.071849,-0.053777,-0.053551,-0.054178,-0.049515,-0.050453,-0.051659,0,3
4,48.128431,-97.276685,2018-08-19,No,-0.053101,-0.038028,-0.046975,-0.052079,-0.084101,-0.050027,...,-0.051531,-0.047736,-0.052661,-0.053492,-0.050059,-0.052688,-0.052509,-0.052358,0,3


,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,vs,bi,fm100,fm1000,erc,etr,pet,vpd,Wildfire_binary,cluster_id
0,34.57238,-118.00486,2022-04-22,No,-0.053101,-0.052751,-0.049151,-0.052082,-0.009310,-0.052693,...,-0.050885,-0.033033,-0.053895,-0.055490,-0.040058,-0.052042,-0.052039,-0.052381,0,0
1,34.57238,-118.00486,2022-04-23,No,-0.053101,-0.060289,-0.062561,-0.052082,0.010894,-0.053345,...,-0.051178,-0.029504,-0.054365,-0.055490,-0.036529,-0.050103,-0.050805,-0.052005,0,0
2,34.57238,-118.00486,2022-04-24,No,-0.053101,-0.068887,-0.065913,-0.052083,0.014262,-0.052871,...,-0.051178,-0.026563,-0.055012,-0.055608,-0.034175,-0.049456,-0.050453,-0.051864,0,0
3,34.57238,-118.00486,2022-04-25,No,-0.053101,-0.073833,-0.068619,-0.052083,0.013080,-0.051627,...,-0.052060,-0.030680,-0.055658,-0.055726,-0.031822,-0.049280,-0.050218,-0.051518,0,0
4,34.57238,-118.00486,2022-04-26,No,-0.053101,-0.074246,-0.066854,-0.052082,-0.043810,-0.049315,...,-0.052119,-0.031857,-0.056070,-0.055843,-0.031234,-0.049809,-0.050688,-0.051500,0,0


In [3]:
train_df["datetime"] = pd.to_datetime(train_df["datetime"])
test_df["datetime"]  = pd.to_datetime(test_df["datetime"])

def add_temporal_features(df):
    df["month"] = df["datetime"].dt.month
    df["dayofyear"] = df["datetime"].dt.dayofyear
    df["year"] = df["datetime"].dt.year
    df["dayofweek"] = df["datetime"].dt.dayofweek

    df["is_summer"] = df["month"].isin([6,7,8]).astype(int)
    df["is_fall"] = df["month"].isin([9,10,11]).astype(int)
    return df

train_df = add_temporal_features(train_df)
test_df  = add_temporal_features(test_df)

weather_cols = [
    'pr','rmax','rmin','sph','srad','tmmn','tmmx','vs',
    'bi','fm100','fm1000','erc','etr','pet','vpd'
]

geo_cols = ['latitude', 'longitude']

feature_cols = geo_cols + weather_cols

In [4]:
# simple find best threshold function
def find_best_threshold(y_true, y_probs):
    best_threshold = 0.5
    best_f1 = 0.0
    for threshold in np.arange(0.0, 1.0, 0.01):
        y_pred = (y_probs >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    return best_threshold, best_f1

In [5]:
# function to train XGBoost model per cluster
def train_xgboost_model(cluster_id):
    print(f"Training model for cluster {cluster_id}...")
    
    # filter data for the specific cluster
    cluster_train_df = train_df[train_df['cluster_id'] == cluster_id]
    cluster_test_df  = test_df[test_df['cluster_id'] == cluster_id]

    n_train = len(cluster_train_df)
    n_test  = len(cluster_test_df)
    print(f"Cluster {cluster_id}: Train rows = {n_train:,}, Test rows = {n_test:,}")

    if n_train == 0 or n_test == 0:
        print(f"Skipping cluster {cluster_id}: no data in train or test.")
        return None
    
    X_train = cluster_train_df[feature_cols]
    y_train = cluster_train_df['Wildfire_binary']
    
    X_test = cluster_test_df[feature_cols]
    y_test = cluster_test_df['Wildfire_binary']

    if y_train.nunique() < 2:
        print(f"Skipping cluster {cluster_id}: only one class in TRAIN (no fires or all fires).")
        return None
    
    # calculate pos_weight to handle class imbalance (same as GBM)
    pos_weight = ((y_train == 0).sum() / (y_train == 1).sum())
    print(f"Positive Class Weight = {pos_weight:.2f}")
    
    # initialize and train the model
    model = XGBClassifier(
        n_estimators=400, # increased number of trees after tuning
        learning_rate=0.05, # satisfactory learning rate after tuning
        max_depth=8, # higher depth for capturing complex patterns
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss', # standard for binary classification
        scale_pos_weight=pos_weight,
        random_state=42, # for reproducibility. Idk why 42 is so popular
        tree_method="hist"
    )
    
    model.fit(X_train, y_train)

    # find the best threshold on the training set  
    y_train_proba = model.predict_proba(X_train)[:, 1]
    threshold, best_f1 = find_best_threshold(y_train, y_train_proba)
    print(f"best threshold on training set: {threshold:.2f} with F1: {best_f1:.4f}")
    
    # make predictions
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_pred_proba)
    
    print(f"Cluster {cluster_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Average Precision: {avg_precision:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # save the model
    joblib.dump(model, f"xgboost_model_cluster_{cluster_id}.pkl")
    
    return {
        'cluster_id': cluster_id,
        'y_test': y_test,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        "roc_auc": roc_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "avg_precision": avg_precision,
        "n_test": len(y_test)
    }

In [ ]:
all_results = []
all_y_true = []
all_y_proba = []
all_y_pred = []

clusters = sorted(train_df["cluster_id"].unique())
print(f"Training models for clusters: {clusters}")
for cid in clusters:
    result = train_xgboost_model(cid)
    if result is None:
        continue
    all_results.append(result)
    all_y_true.append(result['y_test'])
    all_y_proba.append(result['y_pred_proba'])
    all_y_pred.append(result['y_pred'])

Training models for clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


TypeError: train_xgboost_model() got an unexpected keyword argument 'threshold'

In [ ]:
if all_y_true:
    y_true_total  = pd.concat(all_y_true)
    y_proba_total = np.concatenate(all_y_proba)
    y_pred_total  = np.concatenate(all_y_pred)

    # overall evaluation
    overall_accuracy = accuracy_score(y_true_total, y_pred_total)
    overall_roc_auc = roc_auc_score(y_true_total, y_proba_total)
    overall_precision = precision_score(y_true_total, y_pred_total)
    overall_recall = recall_score(y_true_total, y_pred_total)
    overall_f1 = f1_score(y_true_total, y_pred_total)
    overall_avg_precision = average_precision_score(y_true_total, y_proba_total)
    print("\nOverall Model Performance:")
    print(f"Overall Accuracy: {overall_accuracy:.4f}")
    print(f"Overall ROC AUC: {overall_roc_auc:.4f}")
    print(f"Overall Precision: {overall_precision:.4f}")
    print(f"Overall Recall: {overall_recall:.4f}")
    print(f"Overall F1 Score: {overall_f1:.4f}")
    print(f"Overall Average Precision: {overall_avg_precision:.4f}")
    print("Overall Confusion Matrix:")
    print(confusion_matrix(y_true_total, y_pred_total))
    print("Overall Classification Report:")
    print(classification_report(y_true_total, y_pred_total))
    


Overall Model Performance:
Overall Accuracy: 0.6796
Overall ROC AUC: 0.5986
Overall Precision: 0.1628
Overall Recall: 0.3980
Overall F1 Score: 0.2311
Overall Average Precision: 0.1619
Overall Confusion Matrix:
[[910905 357207]
 [105071  69478]]
Overall Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80   1268112
           1       0.16      0.40      0.23    174549

    accuracy                           0.68   1442661
   macro avg       0.53      0.56      0.51   1442661
weighted avg       0.81      0.68      0.73   1442661

